In [2]:
import sys

# ၁။ ပြဿနာဖြစ်နေတဲ့ Langchain အဟောင်းတွေကို အကုန် Force Uninstall လုပ်မယ်
!{sys.executable} -m pip uninstall -y langchain langchain-core langchain-community langchain-openai

# ၂။ ပြီးမှ လိုအပ်တာတွေကို လမ်းကြောင်းမှန်ထဲ အသစ်ပြန်သွင်းမယ်
!{sys.executable} -m pip install -U fastapi uvicorn nest-asyncio pydantic python-dotenv langchain langchain-openai langchain-community faiss-cpu pypdf docx2txt tiktoken

Found existing installation: langchain 1.4.3
Uninstalling langchain-1.4.3:
  Successfully uninstalled langchain-1.4.3
Found existing installation: langchain-core 1.6.7
Uninstalling langchain-core-1.6.7:
  Successfully uninstalled langchain-core-1.6.7
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-openai 1.6.7
Uninstalling langchain-openai-1.6.7:
  Successfully uninstalled langchain-openai-1.6.7
  Using cached langchain-1.4.3-py3-none-any.whl (164 kB)
  Using cached langchain_openai-1.6.7-py3-none-any.whl (127 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
  Using cached langchain_core-1.6.7-py3-none-any.whl (572 kB)



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
%pip install -q langchain-classic


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# =====================================================================
# 1. လိုအပ်သော Packages များကို Install လုပ်ခြင်း
# =====================================================================

import os
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException, UploadFile, File # 💡 YOLO ပုံလက်ခံရန် UploadFile, File ထပ်တိုးထားပါသည်
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# --- LangChain Classic / Agent Imports (Legacy Mode) ---
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# --- LangChain Tools / Core Imports (နေရာအသစ်) ---
from langchain_core.tools import tool
from langchain_core.tools import create_retriever_tool  # 💡 ဤနေရာသို့ ပြောင်းလဲသွားပါသည်

# --- Data & Vector Store Imports ---
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- YOLO & Image Processing Imports (YOLO အတွက် ထပ်တိုး) ---
import cv2
import numpy as np
from ultralytics import YOLO

# =====================================================================
# 2. Jupyter အတွက် ပြင်ဆင်ခြင်း နှင့် API Key ထည့်သွင်းခြင်း
# =====================================================================
# Jupyter ၏ Event Loop ကို FastAPI နှင့် တွဲသုံးနိုင်ရန်
nest_asyncio.apply()

# 🔑 သင့်၏ OpenRouter API Key ကို ဤနေရာတွင် ထည့်ပါ (sk-or-v1-...)
os.environ["OPENAI_API_KEY"] = ""

# 🌐 OpenRouter Base URL သို့ လမ်းကြောင်းပြောင်းခြင်း (အရေးကြီးသည်)
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# =====================================================================
# 2. FastAPI Application စတင်တည်ဆောက်ခြင်း
# =====================================================================
app = FastAPI(title="EcoBin Pyay Advanced API (OpenRouter Version)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)
# In-memory Database
USERS_DB = {
    "user_123": {"name": "Eco User", "points": 3450, "level": 4, "total_co2_saved": 0.0}
}

# =====================================================================
# 3. AI Scanner (YOLO Model) Load လုပ်ခြင်း (YOLO အတွက် ထပ်တိုး)
# =====================================================================
print("⏳ Loading YOLO AI Model...")
try:
    # သင် Fine-tune လုပ်ထားသော Model ရှိပါက လမ်းကြောင်းပြောင်းထည့်ပါ။
    # ယခု အခြေခံ yolov8n.pt ကို သုံးထားပါသည်။
    yolo_model = YOLO("yolov8n.pt") 
    print("✅ YOLO Model Successfully Loaded!")
except Exception as e:
    print("⚠️ YOLO Model ချိတ်ဆက်ရာတွင် အခက်အခဲရှိနေပါသည်:", e)
    yolo_model = None

# =====================================================================
# 4. Knowledge Base တည်ဆောက်ခြင်း (RAG)
# =====================================================================
def setup_knowledge_base():
    data_dir = "./data"
    
    # Folder မရှိပါက ဖန်တီးပြီး Sample Data ထည့်ပေးမည်
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
        with open(os.path.join(data_dir, "default_fact.txt"), "w", encoding="utf-8") as f:
            f.write("EcoBin Pyay is a smart waste management system designed to protect Irrawaddy dolphins and the environment. It uses gamification to encourage recycling.")

    documents = []
    loaders = [
        DirectoryLoader(data_dir, glob="**/*.pdf", loader_cls=PyPDFLoader),
        DirectoryLoader(data_dir, glob="**/*.docx", loader_cls=Docx2txtLoader),
        DirectoryLoader(data_dir, glob="**/*.txt", loader_cls=TextLoader)
    ]
    
    for loader in loaders:
        try:
            documents.extend(loader.load())
        except Exception as e:
            print(f"Warning skipping some files: {e}")

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    docs = text_splitter.split_documents(documents)
    
    embeddings = OpenAIEmbeddings()
    vector_store = FAISS.from_documents(docs, embeddings)
    
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    return retriever

print("⏳ Building Knowledge Base from ./data folder...")
kb_retriever = setup_knowledge_base()
print("✅ Knowledge Base Successfully Built!")

knowledge_base_tool = create_retriever_tool(
    kb_retriever,
    "environmental_knowledge_base",
    "Use this tool to search for official information, environment laws, global warming facts, and details about Irrawaddy dolphins. Translate the concept into Burmese."
)

# =====================================================================
# 5. LangChain Tools များ (Point တွက်ခြင်း၊ ဆုလက်ဆောင် စစ်ဆေးခြင်း)
# =====================================================================
@tool
def get_user_stats(user_id: str) -> str:
    """Check the user's current points, level, and CO2 saved in EcoBin Pyay app."""
    user = USERS_DB.get(user_id)
    if user:
        return f"User '{user['name']}' has {user['points']} points, is at Level {user['level']}, and saved {user['total_co2_saved']} kg of CO2. Tell them if their level is high, their virtual tree is blooming and the dolphin is happy!"
    return "User account not found."

@tool
def get_app_rewards_info() -> str:
    """Get information about app rewards in EcoBin Pyay."""
    return "1. KBZPay Cash (1000 Pts = 1000 MMK). 2. Mate Swe Oway Ride (500 Pts = 500 MMK Off). 3. Shwe Min Tha Mee Shopping Voucher (2000 Pts = 2000 MMK Off)."

@tool
def get_recycling_guide(waste_type: str) -> str:
    """Provide recycling instructions and point values."""
    guides = {
        "plastic": "Empty and crush plastic bottles. Reward: 50 Points.",
        "glass": "Handle glass carefully, do not break. Reward: 100 Points.",
        "paper": "Keep paper dry and fold it. Reward: 20 Points.",
        "can": "Empty and crush aluminum/tin cans. Reward: 80 Points."
    }
    for key in guides:
        if key in waste_type.lower():
            return guides[key]
    return "Ensure waste is clean. Standard general waste does not give points."

# =====================================================================
# 6. AI Agent တည်ဆောက်ခြင်း
# =====================================================================
llm = ChatOpenAI(model="google/gemini-3.8-flash", temperature=0.4)
tools = [get_user_stats, get_app_rewards_info, get_recycling_guide, knowledge_base_tool]

system_instruction = """
You are 'Eco-Coach', an intelligent and Kawaii AI assistant for the 'EcoBin Pyay' smart waste management app in Myanmar.
CORE INSTRUCTIONS:
1. TRANSLATE TO BURMESE: Even if the knowledge base or tools return text in English, YOU MUST GENERATE YOUR FINAL RESPONSE ENTIRELY IN BURMESE (Myanmar Language).
2. TONE: Be very friendly, encouraging, and use Kawaii emojis (🐬, 🌸, 💖, ✨).
3. FORMATTING: You must format your response strictly as a bulleted list. Each point must be on a new line. Do NOT use markdown headers (e.g., #, ##). Do NOT write long paragraphs. Keep each point concise and easy to read.
4. CONTEXT AWARENESS: If the user has a high level (Level 3 or 4+), remind them that their virtual tree is blooming beautifully and their dolphin is very happy.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_instruction),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# =====================================================================
# 7. API Routes (Endpoints) များ
# =====================================================================
class SyncRequest(BaseModel):
    user_id: str
    current_points: int
    level: int
    total_co2: float

class ChatRequest(BaseModel):
    user_id: str
    message: str

class ChatResponse(BaseModel):
    reply: str

@app.post("/api/sync")
async def sync_user_state(request: SyncRequest):
    if request.user_id in USERS_DB:
        USERS_DB[request.user_id].update({
            "points": request.current_points,
            "level": request.level,
            "total_co2_saved": request.total_co2
        })
        return {"status": "success", "message": "User stats synced."}
    
    USERS_DB[request.user_id] = {
        "name": "Eco User",
        "points": request.current_points,
        "level": request.level,
        "total_co2_saved": request.total_co2
    }
    return {"status": "created", "message": "New user synced."}

@app.post("/api/chat", response_model=ChatResponse)
async def chat_with_eco_coach(request: ChatRequest):
    try:
        structured_input = f"[User ID: {request.user_id}] \nUser Message: {request.message}"
        response = agent_executor.invoke({"input": structured_input})
        return ChatResponse(reply=response["output"])
    except Exception as e:
        print(f"Agent Error: {e}")
        raise HTTPException(status_code=500, detail="Eco-Coach is taking a short rest. Please try again.")

# 📸 ယခု ထပ်တိုးထားသော အမှိုက် Scan ဖတ်မည့် YOLO Endpoint
@app.post("/api/scan")
async def scan_waste(file: UploadFile = File(...)):
    if not yolo_model:
        raise HTTPException(status_code=500, detail="AI Scanner (YOLO) ချိတ်ဆက်ထားခြင်းမရှိပါ။")

    try:
        # ပုံကို ဖတ်ယူခြင်း
        contents = await file.read()
        nparr = np.frombuffer(contents, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # YOLO ဖြင့် အမှိုက်ကို စစ်ဆေးခြင်း
        results = yolo_model(img, verbose=False)

        detected_label = "Unknown"
        points = 0
        co2_saved = 0.0

        for r in results:
            for box in r.boxes:
                class_name = yolo_model.names[int(box.cls[0])].lower()

                # အမှိုက်အမျိုးအစားအလိုက် Point သတ်မှတ်ခြင်း
                if class_name in ['bottle', 'plastic', 'plastic bottle']:
                    detected_label = "ပလတ်စတစ်ဘူး"
                    points, co2_saved = 50, 0.08
                    break
                elif class_name in ['cup', 'can', 'metal', 'aluminum can']:
                    detected_label = "အချိုရည် သံဘူး"
                    points, co2_saved = 80, 0.15
                    break
                elif class_name in ['wine glass', 'glass', 'glass bottle']:
                    detected_label = "ဖန်ပုလင်း"
                    points, co2_saved = 100, 0.3
                    break

        if detected_label == "Unknown":
            return {"status": "failed", "message": "အမှိုက်အမျိုးအစားကို သေချာစွာ မဖတ်နိုင်ပါ။ ထပ်မံရိုက်ကူးကြည့်ပါ။"}

        return {
            "status": "success",
            "name": detected_label,
            "pts": points,
            "co2": co2_saved,
            "label": f"{detected_label} (AI Verified)"
        }
    except Exception as e:
        print("Scan Error:", e)
        raise HTTPException(status_code=500, detail="Scan ဖတ်ရာတွင် အခက်အခဲရှိနေပါသည်။")

# =====================================================================
# 8. Server စတင် Run ခြင်း (Jupyter-friendly Asynchronous Way)
# =====================================================================
import asyncio

print("🚀 Starting EcoBin API Server on http://localhost:8000")
print("⚠️ Server ရပ်တန့်လိုပါက Jupyter ၏ 'Interrupt Kernel' (Stop ခလုတ်) ကို နှိပ်ပါ။")

# Uvicorn ကို Jupyter ရဲ့ လက်ရှိ Event Loop ထဲမှာပဲ Run ခိုင်းခြင်း
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

# ဤနေရာတွင် 'await' သုံးခြင်းဖြင့် Error လုံးဝ မတက်တော့ပါ
await server.serve()

⏳ Loading YOLO AI Model...
✅ YOLO Model Successfully Loaded!
⏳ Building Knowledge Base from ./data folder...
✅ Knowledge Base Successfully Built!
🚀 Starting EcoBin API Server on http://localhost:8000
⚠️ Server ရပ်တန့်လိုပါက Jupyter ၏ 'Interrupt Kernel' (Stop ခလုတ်) ကို နှိပ်ပါ။


INFO:     Started server process [18112]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:61174 - "OPTIONS /api/sync HTTP/1.1" 200 OK
INFO:     127.0.0.1:61174 - "POST /api/sync HTTP/1.1" 200 OK
INFO:     127.0.0.1:61184 - "POST /api/sync HTTP/1.1" 200 OK
INFO:     127.0.0.1:61188 - "POST /api/sync HTTP/1.1" 200 OK
INFO:     127.0.0.1:61190 - "OPTIONS /api/chat HTTP/1.1" 200 OK


> Entering new AgentExecutor chain...

Invoking: `get_user_stats` with `{'user_id': 'Poe Myat Thiri'}`


User 'Eco User' has 180 points, is at Level 1, and saved 0.45 kg of CO2. Tell them if their level is high, their virtual tree is blooming and the dolphin is happy!
Invoking: `environmental_knowledge_base` with `{'query': 'EcoBin Pyay bin locations smart bin locations'}`


Country Updates & Response of Myanmar               
to Plastic Waste / E-Waste Amendment 
1 
Dr. Tin Aung Win (Mr.) 
Director 
Environmental Conservation Department  
Ministry of Natural Resources and Environmental Conservation, Myanmar

“Capacity Building on Environmentally Sound Management of Single-
use 

In [ ]:
# =====================================================================
# 1. လိုအပ်သော Packages များကို Install လုပ်ခြင်း
# =====================================================================
import os
import nest_asyncio
import uvicorn
import uuid
from fastapi import FastAPI, HTTPException, UploadFile, File # 💡 YOLO ပုံလက်ခံရန် UploadFile, File ထပ်တိုးထားပါသည်
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# --- LangChain Classic / Agent Imports (Legacy Mode) ---
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# --- LangChain Tools / Core Imports (နေရာအသစ်) ---
from langchain_core.tools import tool
from langchain_core.tools import create_retriever_tool  # 💡 ဤနေရာသို့ ပြောင်းလဲသွားပါသည်

# --- Data & Vector Store Imports ---
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- YOLO & Image Processing Imports (YOLO အတွက် ထပ်တိုး) ---
import cv2
import numpy as np
from ultralytics import YOLO

# --- Roboflow Imports (Cloud AI) ---
from roboflow import Roboflow
from dotenv import load_dotenv

# =====================================================================
# 2. API Keys များကို .env မှ ဖတ်ယူခြင်း
# =====================================================================
nest_asyncio.apply()
load_dotenv() # .env ဖိုင်ထဲမှ Key များကို ဆွဲထုတ်ခြင်း

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

# =====================================================================
# 3. FastAPI Application စတင်တည်ဆောက်ခြင်း
# =====================================================================
app = FastAPI(title="EcoBin Pyay Advanced API (Roboflow Cloud Version)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

USERS_DB = {}

# =====================================================================
# 4. Roboflow AI Scanner ကို လှမ်းချိတ်ခြင်း
# =====================================================================
print("⏳ Loading Roboflow Cloud AI Model...")
try:
    rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
    vision_model = rf.workspace("grad-jhbv7").project("waste-classification-irwkg").version(1).model
    print("✅ Roboflow Model Successfully Connected!")
except Exception as e:
    print("⚠️ Roboflow ချိတ်ဆက်ရာတွင် အခက်အခဲရှိနေပါသည်:", e)
    vision_model = None

# =====================================================================
# 5. Knowledge Base တည်ဆောက်ခြင်း (RAG)
# =====================================================================
def setup_knowledge_base():
    data_dir = "./data"
    
    if not os.path.exists(data_dir):
        os.makedirs(data_dir)
        with open(os.path.join(data_dir, "default_fact.txt"), "w", encoding="utf-8") as f:
            f.write("EcoBin Pyay is a smart waste management system designed to protect Irrawaddy dolphins and the environment. It uses gamification to encourage recycling.")

    documents = []
    loaders = [
        DirectoryLoader(data_dir, glob="**/*.pdf", loader_cls=PyPDFLoader),
        DirectoryLoader(data_dir, glob="**/*.docx", loader_cls=Docx2txtLoader),
        DirectoryLoader(data_dir, glob="**/*.txt", loader_cls=TextLoader)
    ]
    
    for loader in loaders:
        try:
            documents.extend(loader.load())
        except Exception as e:
            pass

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    docs = text_splitter.split_documents(documents)
    
    embeddings = OpenAIEmbeddings()
    if docs:
        vector_store = FAISS.from_documents(docs, embeddings)
        return vector_store.as_retriever(search_kwargs={"k": 3})
    return None

print("⏳ Building Knowledge Base from ./data folder...")
kb_retriever = setup_knowledge_base()
print("✅ Knowledge Base Successfully Built!")

tools = []
if kb_retriever:
    tools.append(create_retriever_tool(
        kb_retriever,
        "environmental_knowledge_base",
        "Use this tool to search for official information, environment laws, global warming facts, and details about Irrawaddy dolphins. Translate the concept into Burmese."
    ))

# =====================================================================
# 6. LangChain Tools များ
# =====================================================================
@tool
def get_user_stats(user_id: str) -> str:
    """Check the user's current points, level, and CO2 saved in EcoBin Pyay app."""
    user = USERS_DB.get(user_id)
    if user:
        return f"User '{user['name']}' has {user['points']} points, is at Level {user['level']}, and saved {user['total_co2_saved']} kg of CO2."
    return "User account not found."

@tool
def get_app_rewards_info() -> str:
    """Get information about app rewards in EcoBin Pyay."""
    return "1. KBZPay Cash (1000 Pts = 1000 MMK). 2. Mate Swe Oway Ride (500 Pts = 500 MMK Off). 3. Shwe Min Tha Mee Shopping Voucher (2000 Pts = 2000 MMK Off)."

tools.extend([get_user_stats, get_app_rewards_info])

# =====================================================================
# 7. AI Agent (Eco-Coach)
# =====================================================================
llm = ChatOpenAI(model="openai/gpt-4o-mini", temperature=0.4)

system_instruction = """
You are 'Eco-Coach', an intelligent and Kawaii AI assistant for the 'EcoBin Pyay' smart waste management app in Myanmar.
CORE INSTRUCTIONS:
1. TRANSLATE TO BURMESE: YOU MUST GENERATE YOUR FINAL RESPONSE ENTIRELY IN BURMESE.
2. TONE: Be very friendly, encouraging, and use Kawaii emojis (🐬, 🌸, 💖).
3. FORMATTING: You must format your response strictly as a bulleted list. Each point must be on a new line. Do NOT use markdown headers (e.g., #, ##). Do NOT write long paragraphs.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_instruction),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False)

# =====================================================================
# 8. API Routes
# =====================================================================
class SyncRequest(BaseModel):
    user_id: str
    current_points: int
    level: int
    total_co2: float

class ChatRequest(BaseModel):
    user_id: str
    message: str

@app.post("/api/sync")
async def sync_user_state(request: SyncRequest):
    USERS_DB[request.user_id] = {
        "name": request.user_id, "points": request.current_points,
        "level": request.level, "total_co2_saved": request.total_co2
    }
    return {"status": "success"}

@app.post("/api/chat")
async def chat_with_eco_coach(request: ChatRequest):
    try:
        response = agent_executor.invoke({"input": f"[User ID: {request.user_id}] \nUser Message: {request.message}"})
        return {"reply": response["output"]}
    except Exception as e:
        raise HTTPException(status_code=500, detail="AI အလုပ်များနေပါသည်။")

# 📸 ပြင်ဆင်ထားသော Roboflow Scan Endpoint (Classification Model အတွက်)
@app.post("/api/scan")
async def scan_waste(file: UploadFile = File(...)):
    if not vision_model:
        raise HTTPException(status_code=500, detail="Roboflow AI နှင့် ချိတ်ဆက်ထားခြင်း မရှိပါ။")

    # ယာယီ File နာမည်တစ်ခုဖန်တီး၍ ပုံကိုသိမ်းခြင်း
    temp_filename = f"temp_{uuid.uuid4()}.jpg"
    with open(temp_filename, "wb") as buffer:
        buffer.write(await file.read())

    try:
        prediction = vision_model.predict(temp_filename).json()
        
        print("==== AI Prediction Result ====")
        print(prediction)
        print("==============================")

        # ယာယီပုံကို ပြန်ဖျက်ခြင်း
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

        # 💡 JSON ထဲမှ class အမည်ကို အမှန်ကန်ဆုံး ဆွဲထုတ်ခြင်း
        class_name = "unknown"
        preds = prediction.get('predictions', [])
        if preds and len(preds) > 0:
            # Log ထဲမှာပါတဲ့ 'top' ကို တိုက်ရိုက်ယူသုံးခြင်း
            if 'top' in preds[0]:
                class_name = preds[0]['top'].lower()
            elif 'predictions' in preds[0] and len(preds[0]['predictions']) > 0:
                class_name = preds[0]['predictions'][0]['class'].lower()

        if class_name == "unknown":
            return {"status": "failed", "message": "အမှိုက်အမျိုးအစားကို သေချာစွာ မဖတ်နိုင်ပါ။"}

        detected_label = "Unknown"
        points = 0
        co2_saved = 0.0

        # အမှိုက်အမျိုးအစားအလိုက် Point သတ်မှတ်ခြင်း
        if "pet" in class_name or "plastic" in class_name or "bottle" in class_name:
            detected_label, points, co2_saved = "ပလတ်စတစ်ဘူး (PET)", 50, 0.08
        elif "can" in class_name or "metal" in class_name or "alu" in class_name:
            detected_label, points, co2_saved = "အချိုရည် သံဘူး", 80, 0.15
        elif "glass" in class_name:
            detected_label, points, co2_saved = "ဖန်ပုလင်း", 100, 0.3
        else:
            detected_label, points, co2_saved = class_name.capitalize(), 10, 0.05

        return {
            "status": "success",
            "name": detected_label,
            "pts": points,
            "co2": co2_saved,
            "label": f"{detected_label} (AI Verified)"
        }
    except Exception as e:
        if os.path.exists(temp_filename):
            os.remove(temp_filename)
        print("Scan Error:", e)
        raise HTTPException(status_code=500, detail="Scan ဖတ်ရာတွင် အခက်အခဲရှိနေပါသည်။")

# =====================================================================
# 9. Server Run ခြင်း
# =====================================================================
import asyncio

print("🚀 Starting EcoBin API Server on http://0.0.0.0:8000")
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

⏳ Loading Roboflow Cloud AI Model...
loading Roboflow workspace...
loading Roboflow project...


version.model is deprecated and will be removed in a future release; use version.models() (all trained models) or version.trainings() instead.


✅ Roboflow Model Successfully Connected!
⏳ Building Knowledge Base from ./data folder...
✅ Knowledge Base Successfully Built!
🚀 Starting EcoBin API Server on http://0.0.0.0:8000


INFO:     Started server process [19268]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:61243 - "OPTIONS /api/sync HTTP/1.1" 200 OK
INFO:     127.0.0.1:61243 - "POST /api/sync HTTP/1.1" 200 OK
==== AI Prediction Result ====
{'predictions': [{'inference_id': '67e4185f-aa37-4247-b5af-738e972d31b1', 'time': 0.00555306626483798, 'resolved_model': {'model_id': 'waste-classification-irwkg/1', 'model_package_id': '5362b72bfb9f01d2e0b8cba2048d932c', 'backend': 'onnx', 'quantization': 'fp32'}, 'image': {'width': 640, 'height': 480}, 'predictions': [{'class': 'PET', 'class_id': 2, 'confidence': 1.0}], 'top': 'PET', 'confidence': 1.0, 'image_path': 'temp_5ded3c31-9143-40da-8919-47701f2c9ba7.jpg', 'prediction_type': 'ClassificationModel'}], 'image': (640, 480)}
INFO:     127.0.0.1:61257 - "POST /api/scan HTTP/1.1" 200 OK
INFO:     127.0.0.1:61257 - "POST /api/sync HTTP/1.1" 200 OK
==== AI Prediction Result ====
{'predictions': [{'inference_id': 'fd65c257-b6d4-41a0-b3a4-65f2f4c4f06d', 'time': 0.005562476348131895, 'resolved_model': {'model_id': 'waste-classificatio